In [20]:
import os

BASE = r"C:/Users/Win10/OneDrive - Università degli Studi di Torino/Desktop/repo_dss/dss_lab_project"
FILLED = BASE + "/dataset/filled"

print(os.listdir(FILLED))


['albums.json', 'artists_filled.json', 'artists_filled_new_ids.json', 'artists_filled_with_feat.json', 'dates.json', 'participations.json', 'tracks_filled.json', 'tracks_filled_date_id.json', 'tracks_filled_new_ids.json']


In [21]:
TRACKS = FILLED + "/tracks_filled_date_id.json"
ARTISTS = FILLED + "/artists_filled_with_feat.json"


import json

with open(TRACKS, "r", encoding="utf-8") as f:
    tracks = json.load(f)

with open(ARTISTS, "r", encoding="utf-8") as f:
    artists = json.load(f)

print("Tracce:", len(tracks))
print("Artisti:", len(artists))


Tracce: 11166
Artisti: 104


In [22]:
# Colonne delle tracce
print("COLONNE TRACKS:")
for col in tracks[0].keys():
    print("-", col)

print("\nCOLONNE ARTISTS:")
for col in artists[0].keys():
    print("-", col)


COLONNE TRACKS:
- id
- id_artist
- title
- featured_artists
- primary_artist
- language
- album
- swear_IT
- swear_EN
- swear_IT_words
- swear_EN_words
- year
- month
- day
- n_sentences
- n_tokens
- char_per_tok
- avg_token_per_clause
- bpm
- rolloff
- flux
- rms
- flatness
- spectral_complexity
- pitch
- loudness
- album_name
- album_release_date
- album_type
- disc_number
- track_number
- duration_ms
- explicit
- popularity
- id_album
- lyrics
- streams@1month
- new_id_album
- new_track_id
- new_id_artist
- date_id

COLONNE ARTISTS:
- id_author
- name
- gender
- birth_date
- birth_place
- nationality
- description
- active_start
- active_end
- province
- region
- country
- latitude
- longitude
- type
- active-end
- new_id_artist


In [23]:
# Rimuovere la colonna 'active-end' da tutti gli artisti
for a in artists:
    if "active-end" in a:
        del a["active-end"]

print("Colonne dopo rimozione:")
print(artists[0].keys())


Colonne dopo rimozione:
dict_keys(['id_author', 'name', 'gender', 'birth_date', 'birth_place', 'nationality', 'description', 'active_start', 'active_end', 'province', 'region', 'country', 'latitude', 'longitude', 'type', 'new_id_artist'])


In [24]:
print("TIPI DELLE COLONNE TRACKS:\n")
sample_track = tracks[0]
for col, val in sample_track.items():
    print(f"{col}: {type(val).__name__}")


TIPI DELLE COLONNE TRACKS:

id: str
id_artist: str
title: str
featured_artists: str
primary_artist: str
language: str
album: str
swear_IT: int
swear_EN: int
swear_IT_words: str
swear_EN_words: str
year: float
month: float
day: float
n_sentences: float
n_tokens: float
char_per_tok: float
avg_token_per_clause: float
bpm: float
rolloff: float
flux: float
rms: float
flatness: float
spectral_complexity: float
pitch: float
loudness: float
album_name: str
album_release_date: str
album_type: str
disc_number: float
track_number: float
duration_ms: float
explicit: bool
popularity: float
id_album: str
lyrics: str
streams@1month: float
new_id_album: str
new_track_id: str
new_id_artist: str
date_id: str


In [25]:
print("\nTIPI DELLE COLONNE ARTISTS:\n")
sample_artist = artists[0]
for col, val in sample_artist.items():
    print(f"{col}: {type(val).__name__}")


TIPI DELLE COLONNE ARTISTS:

id_author: str
name: str
gender: str
birth_date: NoneType
birth_place: str
nationality: NoneType
description: str
active_start: str
active_end: NoneType
province: str
region: str
country: str
latitude: str
longitude: str
type: str
new_id_artist: str


In [26]:
def check_mixed_types(records):
    mixed = {}
    
    # prendi tutte le colonne che compaiono in almeno un record
    all_columns = set()
    for rec in records:
        all_columns.update(rec.keys())
    
    # controlla il tipo dei valori per ciascuna colonna
    for col in all_columns:
        types = set(type(r.get(col)).__name__ for r in records if r.get(col) is not None)
        if len(types) > 1:
            mixed[col] = types

    return mixed

print("Colonne con tipi MISTI nelle TRACCE:")
print(check_mixed_types(tracks))

print("\nColonne con tipi MISTI negli ARTISTI:")
print(check_mixed_types(artists))


Colonne con tipi MISTI nelle TRACCE:
{'month': {'str', 'float'}, 'track_number': {'str', 'float'}, 'day': {'str', 'float'}, 'year': {'str', 'float'}, 'duration_ms': {'int', 'str', 'float'}}

Colonne con tipi MISTI negli ARTISTI:
{'latitude': {'str', 'float'}, 'longitude': {'str', 'float'}}


In [27]:
# tracks 
# Correzione SOLO dei campi con tipi misti
for t in tracks:
    
    # year / month / day → int
    for col in ["year", "month", "day"]:
        val = t.get(col)
        if val is not None:
            try:
                t[col] = int(float(val))
            except:
                t[col] = None

    # track_number → int
    val = t.get("track_number")
    if val is not None:
        try:
            t["track_number"] = int(float(val))
        except:
            t["track_number"] = None

    # duration_ms → int
    val = t.get("duration_ms")
    if val is not None:
        try:
            t["duration_ms"] = int(float(val))
        except:
            t["duration_ms"] = None
 

In [28]:
# artists 
# Correzione SOLO latitude / longitude
for a in artists:
    for col in ["latitude", "longitude"]:
        val = a.get(col)
        if val is not None:
            try:
                a[col] = float(val)
            except:
                a[col] = None


In [29]:
# check again
print("TRACKS:", check_mixed_types(tracks))
print("ARTISTS:", check_mixed_types(artists))
 

TRACKS: {}
ARTISTS: {}


In [30]:
from collections import Counter

def count_nulls(records):
    """
    Conta quanti valori sono None per ogni colonna.
    Ritorna un dizionario {colonna: numero_null} e il totale dei record.
    """
    null_count = Counter()
    total = len(records)

    for rec in records:
        for key, value in rec.items():
            if value is None:
                null_count[key] += 1
    
    return null_count, total


# === TRACCE ===
null_tracks, tot_tracks = count_nulls(tracks)

print("MISSING VALUES NELLE TRACCE:")
for col, n in null_tracks.items():
    perc = round(n / tot_tracks * 100, 2)
    print(f"{col}: {n} ({perc}%)")

print("\nTotale tracce:", tot_tracks)


# === ARTISTI ===
null_artists, tot_artists = count_nulls(artists)

print("\nMISSING VALUES NEGLI ARTISTI:")
for col, n in null_artists.items():
    perc = round(n / tot_artists * 100, 2)
    print(f"{col}: {n} ({perc}%)")

print("\nTotale artisti:", tot_artists)


MISSING VALUES NELLE TRACCE:
featured_artists: 7649 (68.5%)
album: 1514 (13.56%)
month: 1186 (10.62%)
day: 1312 (11.75%)
bpm: 64 (0.57%)
rolloff: 64 (0.57%)
flux: 64 (0.57%)
rms: 64 (0.57%)
flatness: 64 (0.57%)
spectral_complexity: 64 (0.57%)
pitch: 64 (0.57%)
loudness: 64 (0.57%)
year: 440 (3.94%)
language: 105 (0.94%)
album_release_date: 7 (0.06%)
album_type: 78 (0.7%)
disc_number: 78 (0.7%)
track_number: 11 (0.1%)
duration_ms: 7 (0.06%)
explicit: 78 (0.7%)
popularity: 78 (0.7%)
id_album: 78 (0.7%)
album_name: 3 (0.03%)
n_sentences: 76 (0.68%)
n_tokens: 76 (0.68%)
char_per_tok: 76 (0.68%)
avg_token_per_clause: 76 (0.68%)
lyrics: 3 (0.03%)

Totale tracce: 11166

MISSING VALUES NEGLI ARTISTI:
birth_date: 20 (19.23%)
nationality: 33 (31.73%)
active_end: 102 (98.08%)
birth_place: 14 (13.46%)
description: 18 (17.31%)
active_start: 52 (50.0%)
province: 16 (15.38%)
region: 16 (15.38%)
country: 7 (6.73%)
latitude: 7 (6.73%)
longitude: 7 (6.73%)
type: 1 (0.96%)

Totale artisti: 104


In [31]:
# valori sporchi 
def find_dirty_values(records):
    suspicious = {"unknown", "Unknown", "UNKNOWN",
                  "null", "NULL",
                  "na", "NA", "N/A",
                  "", " ", 
                  "none", "None", "NONE",
                  "nan", "NaN", "NAN"}

    dirty = {}

    for rec in records:
        for key, value in rec.items():
            if isinstance(value, str) and value.strip() in suspicious:
                if key not in dirty:
                    dirty[key] = set()
                dirty[key].add(value.strip())

    return dirty


print("VALORI SPORCHI NELLE TRACCE:")
print(find_dirty_values(tracks))

print("\nVALORI SPORCHI NEGLI ARTISTI:")
print(find_dirty_values(artists))
 

VALORI SPORCHI NELLE TRACCE:
{'title': {'UNKNOWN', 'Unknown'}}

VALORI SPORCHI NEGLI ARTISTI:
{}


In [32]:
# correggiamo title 
def clean_title_unknown(records):
    for rec in records:
        title = rec.get("title")
        if isinstance(title, str) and title.strip().lower() == "unknown":
            rec["title"] = "NULL"

clean_title_unknown(tracks)
 

In [33]:
print(find_dirty_values(tracks))


{'title': {'NULL'}}


In [34]:
# count valori distinct per tracce 
def check_distinct_values(records):
    # valori sospetti da segnalare
    suspicious = {"unknown", "Unknown", "UNKNOWN",
                  "na", "NA", "N/A",
                  "", " ", 
                  "none", "None", "NONE",
                  "nan", "NaN", "NAN"}

    report = {}

    for col in records[0].keys():
        distinct = set()
        for rec in records:
            val = rec.get(col)
            if isinstance(val, str):
                distinct.add(val.strip())
            else:
                distinct.add(val)

        # rimuovi None perché lo trasformiamo in "NULL"
        if None in distinct:
            distinct.remove(None)
            distinct.add("NULL")  # uniformazione

        # valori sospetti presenti?
        dirty = {v for v in distinct if isinstance(v, str) and v in suspicious}

        report[col] = {
            "numero_valori_distinti": len(distinct),
            "valori": list(distinct)[:10],   # mostro solo i primi 10
            "contiene_NULL": ("NULL" in distinct),
            "valori_sospetti": list(dirty) if dirty else "OK"
        }

    return report


# --------- TRACKS ---------
tracks_report = check_distinct_values(tracks)

print("=== CHECK DISTINCT TRACKS ===")
for col, info in tracks_report.items():
    print(f"\nColonna: {col}")
    print(" - Distinct count:", info["numero_valori_distinti"])
    print(" - Contiene NULL:", info["contiene_NULL"])
    print(" - Valori sospetti:", info["valori_sospetti"])
    print(" - Esempi valori:", info["valori"])


=== CHECK DISTINCT TRACKS ===

Colonna: id
 - Distinct count: 11093
 - Contiene NULL: False
 - Valori sospetti: OK
 - Esempi valori: ['TR645009', 'TR162676', 'TR364953', 'TR506532', 'TR_555224', 'TR902095', 'TR383577', 'TR464802', 'TR606025', 'TR851898']

Colonna: id_artist
 - Distinct count: 104
 - Contiene NULL: False
 - Valori sospetti: OK
 - Esempi valori: ['ART07469279', 'ART86576759', 'ART57242110', 'ART96068455', 'ART14383873', 'ART31005348', 'ART66932389', 'ART22979236', 'ART24123617', 'ART02449272']

Colonna: title
 - Distinct count: 10520
 - Contiene NULL: True
 - Valori sospetti: OK
 - Esempi valori: ['CHIARA', 'Jammaria', 'Bitch 3.0', 'Al Qaeda', 'I Ragazzi della Zona', 'Rosso Amarena', 'Rovine', 'Ci Penso Dopo', 'Nessuno vale quanto te', 'Che c’è']

Colonna: featured_artists
 - Distinct count: 1741
 - Contiene NULL: True
 - Valori sospetti: OK
 - Esempi valori: ['Sally Cruz', 'Achille Lauro', 'Liv L’Raynge', 'Slesh, Ckrono', 'Vaanya Diva', 'Clementino, Rocco Hunt', 'CaneSe

In [35]:
# count distinct per artisti
artists_report = check_distinct_values(artists)

print("\n=== CHECK DISTINCT ARTISTS ===")
for col, info in artists_report.items():
    print(f"\nColonna: {col}")
    print(" - Distinct count:", info["numero_valori_distinti"])
    print(" - Contiene NULL:", info["contiene_NULL"])
    print(" - Valori sospetti:", info["valori_sospetti"])
    print(" - Esempi valori:", info["valori"])



=== CHECK DISTINCT ARTISTS ===

Colonna: id_author
 - Distinct count: 104
 - Contiene NULL: False
 - Valori sospetti: OK
 - Esempi valori: ['ART07469279', 'ART86576759', 'ART96068455', 'ART57242110', 'ART14383873', 'ART31005348', 'ART66932389', 'ART22979236', 'ART24123617', 'ART02449272']

Colonna: name
 - Distinct count: 104
 - Contiene NULL: False
 - Valori sospetti: OK
 - Esempi valori: ['colle der fomento', 'nerone', 'coez', 'rose villain', 'babaman', 'slait', 'alfa', 'shiva', 'la pina', 'piotta']

Colonna: gender
 - Distinct count: 2
 - Contiene NULL: False
 - Valori sospetti: OK
 - Esempi valori: ['F', 'M']

Colonna: birth_date
 - Distinct count: 85
 - Contiene NULL: True
 - Valori sospetti: OK
 - Esempi valori: ['1995-09-18', '1986-10-17', '1976-09-30', '1990', '1994-08-22', '1976-10-17', '1983-07-11', '1985-08-28', '2000-03-10', '2001-02-19']

Colonna: birth_place
 - Distinct count: 44
 - Contiene NULL: True
 - Valori sospetti: OK
 - Esempi valori: ['San Benedetto del Tronto',

In [36]:
# creazione id 
from uuid import uuid4

# ============================
# 1) GENERA ID LYRICS PER OGNI TRACK
# ============================
for t in tracks:
    t["new_id_lyrics"] = str(uuid4())


# ============================
# 2) GENERA ID SYMPHONY PER OGNI TRACK
# ============================
for t in tracks:
    t["new_id_symphony"] = str(uuid4())


# ============================
# 3) GENERA ID GEOGRAPHY PER OGNI ARTIST
# ============================
for a in artists:
    a["new_id_geo"] = str(uuid4())


In [37]:

print(json.dumps(tracks[0], indent=4, ensure_ascii=False))
print(json.dumps(artists[0], indent=4, ensure_ascii=False))


{
    "id": "TR934808",
    "id_artist": "ART04205421",
    "title": "​polka 2 :-/",
    "featured_artists": "Ernia, Guè",
    "primary_artist": "Rosa Chemical",
    "language": "pl",
    "album": "FOREVER AND EVER",
    "swear_IT": 13,
    "swear_EN": 6,
    "swear_IT_words": "['cazzo', 'cesso', 'coglioni', 'figa', 'merda', 'palle', 'piscio', 'porca', 'stronzo']",
    "swear_EN_words": "['bitch', 'fuck', 'porno', 'pussy']",
    "year": 2021,
    "month": 4,
    "day": 2,
    "n_sentences": 102.0,
    "n_tokens": 911.0,
    "char_per_tok": 4.1704545455,
    "avg_token_per_clause": 8.1339285714,
    "bpm": 135.32,
    "rolloff": 2895.7767,
    "flux": 1.4499,
    "rms": 0.1786,
    "flatness": 0.8202,
    "spectral_complexity": 25.7148,
    "pitch": 2311.1779,
    "loudness": 17.8675,
    "album_name": "FOREVER AND EVER",
    "album_release_date": "2021-04-09",
    "album_type": "album",
    "disc_number": 1.0,
    "track_number": 3,
    "duration_ms": 207761,
    "explicit": true,
    

In [38]:
"""
# SALVATAGGIO DATASET FINALI IN JSON

output_dir = r"C:\Users\Win10\OneDrive - Università degli Studi di Torino\Desktop\repo_dss\dss_lab_project\dataset"
tracks_path = os.path.join(output_dir, "tracks_final.json")
artists_path = os.path.join(output_dir, "artists_final.json")

with open(tracks_path, "w", encoding="utf-8") as f:
    json.dump(tracks, f, ensure_ascii=False, indent=4)

with open(artists_path, "w", encoding="utf-8") as f:
    json.dump(artists, f, ensure_ascii=False, indent=4)

print("✔ File salvati correttamente")

"""

SyntaxError: (unicode error) 'unicodeescape' codec can't decode bytes in position 56-57: truncated \UXXXXXXXX escape (1659354956.py, line 1)

In [ ]:
PARTECIPATION = FILLED + "/participations.json"



import json

with open(PARTECIPATION, "r", encoding="utf-8") as f:
    partecipation = json.load(f)

print("partecipation:", len(partecipation))



partecipation: 12622


In [ ]:
# Colonne delle partecipation
print("COLONNE partecipation:")
for col in partecipation[0].keys():
    print("-", col)

COLONNE partecipation:
- new_id_artist
- new_track_id
- IsPrimary


In [ ]:
# creiamo surrogate 